In [71]:
import pandas as pd
import numpy as np
import pickle

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import accuracy_score, classification_report
from sklearn.utils.class_weight import compute_sample_weight

from xgboost import XGBClassifier

In [72]:
def load_geo_dataset(file_path):
    # Load data
    df_raw = pd.read_csv(file_path, sep="\t", comment="!", low_memory=False)

    # Fix structure
    df_raw = df_raw.rename(columns={df_raw.columns[0]: "ID_REF"})
    df_raw = df_raw.set_index("ID_REF")
    df = df_raw.T

    # Extract labels
    labels = []
    with open(file_path, "r") as f:
        for line in f:
            if line.startswith("!Sample_characteristics_ch1"):
                parts = line.strip().split("\t")
                labels = [p.split(":")[-1].replace('"', '').strip() for p in parts[1:]]
                break

    df["target"] = labels

    # Clean labels
    df["target"] = df["target"].str.upper().str.strip()

    # Binary mapping (recommended)
    df["target"] = df["target"].map({
        "CTL": 0,
        "MCI": 0,
        "AD": 1
    })

    df = df.dropna(subset=["target"])
    df["target"] = df["target"].astype(int)

    return df

In [73]:
df1 = load_geo_dataset("GSE63060_series_matrix.txt")
df2 = load_geo_dataset("GSE63061_series_matrix.txt")

print("Dataset1:", df1.shape)
print("Dataset2:", df2.shape)

Dataset1: (329, 38324)
Dataset2: (382, 32050)


In [74]:
common_genes = list(set(df1.columns) & set(df2.columns))
common_genes.remove("target")

df1 = df1[common_genes + ["target"]]
df2 = df2[common_genes + ["target"]]

print("Common genes:", len(common_genes))

Common genes: 25549


In [75]:
def normalize(df):
    features = df.drop(columns=["target"])
    features = (features - features.mean()) / features.std()
    features = features.fillna(0)
    df_norm = features.copy()
    df_norm["target"] = df["target"]
    return df_norm

df1 = normalize(df1)
df2 = normalize(df2)

In [76]:
df = pd.concat([df1, df2], axis=0)
df = df.sample(frac=1, random_state=42)

print("Combined dataset:", df.shape)

Combined dataset: (711, 25550)


In [83]:
X = df.drop(columns=["target"])
y = df["target"]

In [90]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [91]:
from sklearn.feature_selection import SelectKBest, f_classif

selector = SelectKBest(f_classif, k=500)

selector.fit(X_train, y_train)

selected_features = X_train.columns[selector.get_support()]

X_train = selector.transform(X_train)
X_test  = selector.transform(X_test)

print("Selected features:", len(selected_features))

Selected features: 500


In [92]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

In [93]:
from sklearn.utils.class_weight import compute_sample_weight

sample_weights = compute_sample_weight(
    class_weight="balanced",
    y=y_train
)

In [94]:
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=700,
    max_depth=6,
    learning_rate=0.02,
    subsample=0.9,
    colsample_bytree=0.9,
    gamma=0.2,
    reg_alpha=0.2,
    reg_lambda=1.5,
    random_state=42,
    eval_metric="logloss"
)

model.fit(
    X_train,
    y_train,
    sample_weight=sample_weights,
    eval_set=[(X_test, y_test)],
    early_stopping_rounds=50,
    verbose=False
)

C:\Users\junai\AppData\Roaming\Python\Python312\site-packages\xgboost\sklearn.py:889: UserWarning: `early_stopping_rounds` in `fit` method is deprecated for better compatibility with scikit-learn, use `early_stopping_rounds` in constructor or`set_params` instead.
  warnings.warn(


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.9, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, gamma=0.2, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.02, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=700,
              n_jobs=None, num_parallel_tree=None, random_state=42, ...)

In [95]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.6783216783216783
              precision    recall  f1-score   support

           0       0.70      0.80      0.75        86
           1       0.62      0.49      0.55        57

    accuracy                           0.68       143
   macro avg       0.66      0.65      0.65       143
weighted avg       0.67      0.68      0.67       143

